In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:

def calculate_loss(model, x_boundary, u_boundary, x_physics):
    u_b_pred = model(x_boundary)
    loss_b = nn.MSELoss()(u_b_pred, u_boundary)

    x_physics.requires_grad_(True)
    u = model(x_physics)
    
    #first derivative
    du_dx = torch.autograd.grad(u, x_physics, 
                                grad_outputs=torch.ones_like(u), 
                                create_graph=True)[0]
    
    # second derivative
    d2u_dx2 = torch.autograd.grad(du_dx, x_physics, 
                                  grad_outputs=torch.ones_like(du_dx), 
                                  create_graph=True)[0]
    
    # Residual for d^2u/dx^2 = -sin(x) -> d^2u/dx^2 + sin(x) = 0
    residual = d2u_dx2 + torch.sin(x_physics)
    loss_pde = torch.mean(residual**2)

    return loss_b + loss_pde



In [ ]:
device = torch.device("cuda")
model = PINN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

#boundary set:
x_b = torch.tensor([[0.0], [np.pi]], dtype=torch.float32).to(device)
u_b = torch.tensor([[0.0], [0.0]], dtype=torch.float32).to(device)

x_p = torch.linspace(0, np.pi, 100).view(-1, 1).to(device)

# trainnn
for epoch in range(5000):
    optimizer.zero_grad()
    loss = calculate_loss(model, x_b, u_b, x_p)
    loss.backward()
    optimizer.step()
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")


In [ ]:
# visualize
with torch.no_grad():
    x_test = torch.linspace(0, np.pi, 100).view(-1, 1).to(device)
    u_pred = model(x_test).cpu().numpy()
    u_true = np.sin(x_test.cpu().numpy())

plt.plot(x_test.cpu(), u_true, label='Exact (sin(x))', linestyle='dashed', lw=3)
plt.plot(x_test.cpu(), u_pred, label='PINN Prediction')
plt.legend()
plt.title("PINN solving $u''(x) = -\sin(x)$")
plt.show()